In [12]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        chosen_move = moves[best_move]
        item_id = chosen_move[1]
        prev_bin = s.current[item_id]
        apply!(s.current, chosen_move)
        push!(s.tabu_buffer, invert_move(p, moves[best_move], prev_bin))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x::Vector{Int})
    profit = 0
    for i in eachindex(x)
        if x[i] > 0
            profit += p.profits[i]
        end
    end
    return -profit
end


function apply!(x, move::Tuple{Int,Int})
    item_idx, new_bin = move
    x[item_idx] = new_bin
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Int,Int}, old_bin::Int)
    return (move[1], old_bin)
end


function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities)) #sum(p.weights .* x)
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # możemy przenieść i do innego plecaka
            if x[i] != k && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i, k))
            end
        end
    end

    return move_list
end



possible_moves (generic function with 1 method)

In [9]:

function generate_problem(n_knapsacks = 3, n_items = 100)
    capacities = rand(DiscreteUniform(2500, 3000), n_knapsacks)
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(capacities, profits, weights)
end

kp1 = generate_problem()


KnapsackProblem([2506, 2939, 2813], [228, 723, 844, 276, 991, 305, 970, 531, 301, 20  …  28, 409, 68, 367, 383, 555, 633, 629, 16, 668], [84, 16, 89, 73, 82, 79, 90, 70, 92, 50  …  76, 16, 66, 94, 47, 24, 75, 76, 83, 16])

In [15]:

function test(kp)
    x0 = fill(0, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)

    println("Pełne rozwiązanie: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end

test(kp1)

Pełne rozwiązanie: [1, 0, 3, 0, 0, 0, 2, 0, 1, 1, 0, 2, 2, 0, 0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 3, 1, 0, 0, 0, 0, 0, 0, 1, 2, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 1, 0, 0, 0, 0, 3, 0]
Best objective: -1956
Last iteration: 1000000
